In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor
import numpy as np
from joblib import dump
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parents[1]
COMBINED_DIR = BASE_DIR / "Data" / "combined"
MODEL_DIR = BASE_DIR / "src" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("COMBINED_DIR exists:", COMBINED_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())

BASE_DIR: /Users/m.mughees/Desktop/Pi-515-2026
COMBINED_DIR exists: True
MODEL_DIR exists: True


In [3]:
combined_train_df = pd.read_csv(COMBINED_DIR / "combined_train.csv")
smos_test_df = pd.read_csv(COMBINED_DIR / "smos_test_aligned.csv")
scan_holdout_df = pd.read_csv(COMBINED_DIR / "scan_holdout_aligned.csv")

print("Combined train:", combined_train_df.shape)
print("SMOS test:", smos_test_df.shape)
print("SCAN holdout:", scan_holdout_df.shape)

combined_train_df.head()

Combined train: (2103894, 11)
SMOS test: (10000, 11)
SCAN holdout: (452, 11)


,latitude,longitude,air_temperature,dewpoint_temperature,total_precipitation,year,month,day,soil_temperature,soil_moisture,source
0,-0.776986,1.426464,-1.215082,-0.989945,-0.367620,-0.314381,-1.292227,-1.323847,-1.369474,0.360234,SMOS
1,-1.216539,0.035936,0.823547,0.796616,-0.359515,0.372430,0.014939,1.733197,1.056937,0.213469,SMOS
2,-1.028159,1.205244,0.467160,0.505021,-0.336933,-1.230129,-0.965435,1.053854,0.334309,0.166913,SMOS
3,0.478879,-0.912151,-0.537826,-0.400612,-0.368519,-1.230129,1.322104,-1.663519,-0.674495,0.199984,SMOS
4,-0.400226,0.035936,0.926438,1.085233,-0.078794,-1.459067,0.014939,-0.191609,0.985257,0.214700,SMOS


In [4]:
features = [
    "latitude",
    "longitude",
    "air_temperature",
    "dewpoint_temperature",
    "total_precipitation",
    "year",
    "month",
    "day",
]

target = "soil_temperature"

In [5]:
X_full = combined_train_df[features]
y_full = combined_train_df[target]

X_train, X_dev, y_train, y_dev = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42
)

X_smos_test = smos_test_df[features]
y_smos_test = smos_test_df[target]

X_scan_test = scan_holdout_df[features]
y_scan_test = scan_holdout_df[target]

print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_smos_test:", X_smos_test.shape)
print("  X_scan_test:", X_scan_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_smos_test:", y_smos_test.shape)
print("  y_scan_test:", y_scan_test.shape)

✅ Data Split Shapes:
  X_train: (1683115, 8)
  X_dev: (420779, 8)
  X_smos_test: (10000, 8)
  X_scan_test: (452, 8)
  y_train: (1683115,)
  y_dev: (420779,)
  y_smos_test: (10000,)
  y_scan_test: (452,)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def identity_function(X):
    return X

def create_combined_soil_temp_pipeline():
    return Pipeline(steps=[
        ("identity", FunctionTransformer(identity_function))
    ])

In [13]:
def evaluate_xgb(X_train, y_train, X_dev, y_dev):
    print("Evaluating XGBoost Regressor for Combined Soil Temperature...")

    param_grid = {
        "algo__n_estimators": [100, 300, 500],
        "algo__max_depth": [3, 5],
        "algo__learning_rate": [0.05, 0.1],
        "algo__subsample": [0.8, 1.0],
    }

    pipeline = create_combined_soil_temp_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ("preprocessor", pipeline),
        ("algo", XGBRegressor(objective="reg:squarederror", random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo,
        param_grid,
        cv=3,
        scoring="neg_mean_squared_error",
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_estimator = grid_search.best_estimator_

    try:
        model = best_estimator.named_steps["algo"]
        feature_names = X_train.columns
        importances = model.feature_importances_

        feature_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop Features:")
        print(feature_df.head(10))

    except Exception as e:
        print("Could not extract feature importances:", e)

    y_pred = best_estimator.predict(X_dev)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    print("\nGrid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, rmse, mae, r2

In [14]:
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)

    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return rmse, mae, r2

In [15]:
print("\n🚀 Evaluating model for: Combined Soil Temperature")

best_model, _, _, _ = evaluate_xgb(X_train, y_train, X_dev, y_dev)


🚀 Evaluating model for: Combined Soil Temperature
Evaluating XGBoost Regressor for Combined Soil Temperature...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Top Features:
                Feature  Importance
2       air_temperature    0.543490
1             longitude    0.336463
0              latitude    0.052624
6                 month    0.040112
5                  year    0.014517
7                   day    0.005538
3  dewpoint_temperature    0.004466
4   total_precipitation    0.002791

Grid searching is done!
Best score (neg MSE): -0.027998651117134007
Best hyperparameters:
{'algo__learning_rate': 0.1, 'algo__max_depth': 5, 'algo__n_estimators': 500, 'algo__subsample': 1.0}


In [16]:
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)
y_smos_test_pred = best_model.predict(X_smos_test)
y_scan_test_pred = best_model.predict(X_scan_test)

In [17]:
train_rmse, train_mae, train_r2 = evaluate_metrics(y_train, y_train_pred, "Train")
dev_rmse, dev_mae, dev_r2 = evaluate_metrics(y_dev, y_dev_pred, "Dev")
smos_test_rmse, smos_test_mae, smos_test_r2 = evaluate_metrics(y_smos_test, y_smos_test_pred, "SMOS Test")
scan_test_rmse, scan_test_mae, scan_test_r2 = evaluate_metrics(y_scan_test, y_scan_test_pred, "SCAN Holdout")


📊 Train Set Performance:
Mean of y_train: 0.0182
RMSE: 0.1627
MAE: 0.1168
R²: 0.9793

📊 Dev Set Performance:
Mean of y_dev: 0.0193
RMSE: 0.1648
MAE: 0.1170
R²: 0.9789

📊 SMOS Test Set Performance:
Mean of y_smos test: 0.0005
RMSE: 0.1544
MAE: 0.1154
R²: 0.9768

📊 SCAN Holdout Set Performance:
Mean of y_scan holdout: 9.1181
RMSE: 1.4193
MAE: 1.0415
R²: 0.9633


In [18]:
results_smos_df = pd.DataFrame({
    "actual_soil_temp": y_smos_test.values,
    "predicted_soil_temp": y_smos_test_pred,
    "source": "SMOS"
})

results_scan_df = pd.DataFrame({
    "actual_soil_temp": y_scan_test.values,
    "predicted_soil_temp": y_scan_test_pred,
    "source": "SCAN"
})

pd.concat([results_smos_df.head(5), results_scan_df.head(5)], ignore_index=True)

,actual_soil_temp,predicted_soil_temp,source
0,0.566021,0.561971,SMOS
1,0.523393,0.442115,SMOS
2,-1.119452,-1.106147,SMOS
3,-1.264064,-1.323233,SMOS
4,-1.571763,-1.505264,SMOS
5,9.143519,7.612933,SCAN
6,16.898148,16.242929,SCAN
7,0.370370,1.119673,SCAN
8,14.212963,15.952738,SCAN
9,0.300926,0.946703,SCAN


In [19]:
dump(best_model, MODEL_DIR / "combined_soil_temperature_model.joblib")

print("✅ Model saved as:", MODEL_DIR / "combined_soil_temperature_model.joblib")

✅ Model saved as: /Users/m.mughees/Desktop/Pi-515-2026/src/models/combined_soil_temperature_model.joblib


In [20]:
metrics_df = pd.DataFrame([
    {"set": "train", "rmse": train_rmse, "mae": train_mae, "r2": train_r2},
    {"set": "dev", "rmse": dev_rmse, "mae": dev_mae, "r2": dev_r2},
    {"set": "smos_test", "rmse": smos_test_rmse, "mae": smos_test_mae, "r2": smos_test_r2},
    {"set": "scan_holdout", "rmse": scan_test_rmse, "mae": scan_test_mae, "r2": scan_test_r2},
])

all_results_df = pd.concat([results_smos_df, results_scan_df], ignore_index=True)

metrics_df.to_csv(MODEL_DIR / "combined_soil_temp_metrics.csv", index=False)
all_results_df.to_csv(MODEL_DIR / "combined_soil_temp_predictions.csv", index=False)

print("✅ Metrics and predictions saved.")

✅ Metrics and predictions saved.


In [21]:
metrics_df

,set,rmse,mae,r2
0,train,0.162714,0.116786,0.979326
1,dev,0.164838,0.117050,0.978874
2,smos_test,0.154361,0.115366,0.976843
3,scan_holdout,1.419266,1.041472,0.963299
